# Feature Regularisation on a Logistic-Regression Head

**Where this comes from.** The linear probe showed that a logistic regression on the OOB-honest tree bits alone scores **0.8467** on credit, against a LightGBM ceiling of **0.8453** and a 2.44M-parameter TabResNet's **0.8475**. The network was contributing nothing. So the head is now fixed as a **logistic regression**, and all experimentation moves to the **features**.

### The four things being tested
| part | what it does | the advisor's words |
|---|---|---|
| C | flips on every bit, at four strengths | *"adding noise at various levels"* |
| D | flip probability grows with depth | *"as the depth increases the perturbations should increase with depth"* |
| E | delete the deepest layers, ramp the flips over what survives | *"delete last few layers, and in the rest ..."* |
| F | keep only depths 0, 1 and 2 | *"keep one experiment where we only keep the first 3 layers"* |

### Two design decisions that make the results readable
**Bits only, no raw features.** Every perturbation arm runs on `tree` alone. If the raw 10 columns were included, the model would have an unperturbed shortcut to fall back on (they reach 0.7785 by themselves), and that would mask whatever the noise does to the bits. One `x+tree` control is included for continuity.

**The weight decay has to be calibrated first (Part B).** For a linear head, weight decay *is* the regularisation. scikit-learn's best setting was `C = 1e-3`, which corresponds to an L2 coefficient near **0.045** on the mean loss, roughly 45x stronger than the `1e-3` we have used for every TabResNet run. Starting the perturbation arms at the old value would put them all on an under-regularised baseline, and we would be measuring the wrong thing. This is the same mistake the sigmoid-softening experiment made, so Part B calibrates before anything else runs.

### The deletion arms get a converged answer, not an SGD one
Deleting bits is a static change to the input, so Part A does it with scikit-learn: convex, solved to the global optimum, no learning rate and no epoch count to defend. Only the *noise* arms need SGD, because the flips are resampled every batch.

⏱ **about an hour.** Runtime → GPU → Run all. Everything is saved to Google Drive.

In [ ]:
# 1 · mount Drive + get the code
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, json
DRIVE = '/content/drive/MyDrive/TKCE/feature_reg'
os.makedirs(DRIVE, exist_ok=True)
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull
print('results ->', DRIVE)

In [ ]:
# 2 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 2b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
import tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('credit already cached — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

def sync(sub):
    """Copy one result folder to Drive once it is finished."""
    if os.path.isdir(sub):
        shutil.copytree(sub, f'{DRIVE}/{os.path.basename(sub)}', dirs_exist_ok=True)
        print('  synced ->', f'{DRIVE}/{os.path.basename(sub)}')

## Part A — deletion, answered by a converged logistic regression
Four static inputs, each solved to the global optimum with the C grid swept and chosen on validation. No SGD is involved, so nothing here can be blamed on optimisation.

| arm | bits kept |
|---|---|
| all layers | ~5,400 at depths 0-5 |
| drop layer 5 | ~2,940 at depths 0-4 |
| drop layers 4, 5 | ~1,480 at depths 0-3 |
| **keep 0, 1, 2 only** | **~700 at depths 0-2** |


In [ ]:
# 3 · Part A — converged sklearn, tree-only, four static inputs (~20 min)
P = '--task 361055 --views tree --encoding oob --C 1e-4 3e-4 1e-3 3e-3 0.01 0.03'
!python -u run_linear_probe.py {P}                      --out results/featreg/A1_all_layers
!python -u run_linear_probe.py {P} --drop-layers 5      --out results/featreg/A2_drop_L5
!python -u run_linear_probe.py {P} --drop-layers 4,5    --out results/featreg/A3_drop_L45
!python -u run_linear_probe.py {P} --keep-layers 0,1,2  --out results/featreg/A4_keep_L012
for d in sorted(glob.glob('results/featreg/A*')): sync(d)

## Part B — calibrate the weight decay of the SGD linear head
The noise arms need SGD, so the SGD linear head must first be regularised as well as the converged one. This sweep finds the weight decay whose validation AUC matches Part A. **Whatever wins here is used for every arm in Parts C to F.**

In [ ]:
# 4 · Part B — weight-decay calibration, no perturbation (~10 min)
LIN = ('--task 361055 --fusion linear --views tree --encoding oob --ensemble 2 '
       '--epochs 400 --dropout 0 --l1 0 --lr 3e-4 --batch-size 128 --device auto '
       '--log-batches')
for wd in ['1e-3', '1e-2', '3e-2', '1e-1', '3e-1']:
    !python -u run_fusion.py {LIN} --weight-decay {wd} --out results/featreg/B_wd_{wd}
for d in sorted(glob.glob('results/featreg/B_wd_*')): sync(d)

In [ ]:
# 5 · pick the calibrated weight decay (best validation AUC) and keep it in WD
import pandas as pd
rows = []
for d in sorted(glob.glob('results/featreg/B_wd_*')):
    js = [f for f in glob.glob(d + '/fusion_*.json') if 'epoch' not in f and 'batch' not in f]
    if not js: continue
    s = json.load(open(js[0])); r = s['results'][0]
    rows.append(dict(wd=s['weight_decay'], val=r['best_val_auc'], test=r['test_auc']))
b = pd.DataFrame(rows).sort_values('wd')
print(b.round(4).to_string(index=False))
WD = f"{b.loc[b.val.idxmax(), 'wd']:g}"
print(f'\ncalibrated weight decay = {WD}  (val {b.val.max():.4f}, test '
      f"{b.loc[b.val.idxmax(), 'test']:.4f})")
a1 = [f for f in glob.glob('results/featreg/A1_all_layers/linear_probe_*.json')]
if a1:
    ref = [r for r in json.load(open(a1[0]))['results'] if r['view'] == 'tree'][0]
    print(f"converged sklearn reference: val {ref['val_auc']:.4f}, test {ref['test_auc']:.4f}")
    print('the SGD head is calibrated if these validation numbers are close')

## Parts C to F — the perturbation arms, all at the calibrated weight decay
Flips are binary (0 becomes 1 and back), resampled every batch, applied during training only. Evaluation always sees clean bits. There is no sigmoid anywhere, so the softening confound that spoiled the first noise experiment cannot occur.

**C, noise at various levels.** Uniform flip probability on every bit: 0.05, 0.1, 0.2, 0.4.

**D, flips increasing with depth.** `p(depth) = P x depth / 5`, so root splits are never touched and the deepest layer gets the full `P`. Each ramp is paired with a uniform control carrying the same average flip rate, which is `0.8 x P`, so the *shape* can be separated from the *amount*.

**E, the combined design.** Delete the deepest layers, then ramp the flips over what remains. A deleted bit is never flipped and stays zero, which the run header confirms by printing `p=0.000` at the deleted depths.

**F, keep only depths 0, 1, 2.** With and without a ramp over those three layers.

In [ ]:
# 6 · Part C — noise at various levels, uniform over all bits
for p in ['0.05', '0.1', '0.2', '0.4']:
    !python -u run_fusion.py {LIN} --weight-decay {WD} --flip-uniform {p} --out results/featreg/C_unif_{p}
for d in sorted(glob.glob('results/featreg/C_unif_*')): sync(d)

In [ ]:
# 7 · Part D — flips increasing with depth, each with a matched-budget uniform control
#     mean p over all bits for a ramp of P is 0.8*P, so those are the controls
for P, ctrl in [('0.1', '0.08'), ('0.2', '0.16'), ('0.5', '0.4')]:
    !python -u run_fusion.py {LIN} --weight-decay {WD} --flip-ramp {P} --out results/featreg/D_ramp_{P}
    print(f'(matched control for ramp {P} is the uniform {ctrl} arm)')
!python -u run_fusion.py {LIN} --weight-decay {WD} --flip-uniform 0.08 --out results/featreg/C_unif_0.08
!python -u run_fusion.py {LIN} --weight-decay {WD} --flip-uniform 0.16 --out results/featreg/C_unif_0.16
for d in sorted(glob.glob('results/featreg/D_ramp_*')) + sorted(glob.glob('results/featreg/C_unif_0.0*')): sync(d)

In [ ]:
# 8 · Part E — delete the deepest layers, then ramp the flips over what survives
!python -u run_fusion.py {LIN} --weight-decay {WD} --deep-layers 5 --deep-delete --out results/featreg/E1_dropL5
!python -u run_fusion.py {LIN} --weight-decay {WD} --deep-layers 5 --deep-delete --flip-ramp 0.2 --out results/featreg/E2_dropL5_ramp02
!python -u run_fusion.py {LIN} --weight-decay {WD} --deep-layers 4,5 --deep-delete --out results/featreg/E3_dropL45
!python -u run_fusion.py {LIN} --weight-decay {WD} --deep-layers 4,5 --deep-delete --flip-ramp 0.2 --out results/featreg/E4_dropL45_ramp02
# same ramp rescaled so the deepest SURVIVING layer reaches the full P
!python -u run_fusion.py {LIN} --weight-decay {WD} --deep-layers 4,5 --deep-delete --flip-ramp 0.2 --flip-ramp-rel --out results/featreg/E5_dropL45_ramp02rel
for d in sorted(glob.glob('results/featreg/E*')): sync(d)

In [ ]:
# 9 · Part F — keep only the first three layers (depths 0, 1, 2)
!python -u run_fusion.py {LIN} --weight-decay {WD} --keep-layers 0,1,2 --out results/featreg/F1_keepL012
!python -u run_fusion.py {LIN} --weight-decay {WD} --keep-layers 0,1,2 --flip-ramp 0.2 --flip-ramp-rel --out results/featreg/F2_keepL012_ramp
# and the x+tree control, for continuity with every earlier experiment
!python -u run_fusion.py {LIN} --weight-decay {WD} --views "tree;x+tree" --out results/featreg/F3_control_xtree
for d in sorted(glob.glob('results/featreg/F*')): sync(d)

In [ ]:
# 10 · summary table across every arm
rows = []
for d in sorted(glob.glob('results/featreg/*')):
    pj = glob.glob(d + '/linear_probe_*.json')
    if pj:                                    # Part A, converged sklearn
        s = json.load(open(pj[0]))
        for r in s['results']:
            rows.append(dict(arm=os.path.basename(d), kind='sklearn (converged)',
                             view=r['view'], bits=r['n_features'],
                             kept_depths=str(s.get('kept_depths')),
                             flip_mean_p=0.0, train_auc=r['train_auc'],
                             val_auc=r['val_auc'], test_auc=r['test_auc'],
                             ceiling=s['tree_ceiling']))
        continue
    fj = [f for f in glob.glob(d + '/fusion_*.json') if 'epoch' not in f and 'batch' not in f]
    if not fj: continue
    s = json.load(open(fj[0]))
    e = pd.read_csv(glob.glob(d + '/fusion_*_epochs.csv')[0])
    for r in s['results']:
        g = e[e.model == r['model']].groupby('epoch').train_auc.mean()
        rows.append(dict(arm=os.path.basename(d), kind='SGD linear',
                         view=r['views'],
                         bits=s.get('kept_n_bits', s.get('tree_encoding_width')),
                         kept_depths=str(s.get('kept_depths')),
                         flip_mean_p=round(s.get('flip_mean_p', 0.0), 4),
                         train_auc=round(g.iloc[-1], 4),
                         val_auc=r['best_val_auc'], test_auc=r['test_auc'],
                         ceiling=s['tree_ceiling']))
t = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print(t.round(4).to_string(index=False))
CEIL = t.ceiling.iloc[0]
print(f'\nbest tree (LightGBM): {CEIL:.4f}   |   luck zone on 2,400 test rows: +/-0.01')
t.to_csv(f'{DRIVE}/featreg_summary.csv', index=False)

In [ ]:
# 11 · two pictures: score by arm, and the flip profile by depth
import matplotlib.pyplot as plt, numpy as np
BLUE, ORANGE, AQUA, VIOLET = '#2a78d6', '#eb6834', '#1baf7a', '#4a3aa7'
MUTED, GRID = '#52514e', '#c8c8c4'
st = t[t.view == 'tree'].copy()
fam = lambda a: ('A deletion (converged)' if a.startswith('A') else
                 'B calibration' if a.startswith('B') else
                 'C uniform noise' if a.startswith('C') else
                 'D depth ramp' if a.startswith('D') else
                 'E delete + ramp' if a.startswith('E') else 'F keep 0-2')
st['family'] = st.arm.map(fam)
cols = {'A deletion (converged)': BLUE, 'B calibration': MUTED,
        'C uniform noise': ORANGE, 'D depth ramp': AQUA,
        'E delete + ramp': VIOLET, 'F keep 0-2': '#b5730f'}
fig, ax = plt.subplots(2, 1, figsize=(13, 10), gridspec_kw={'height_ratios': [1.35, 1]})
a = ax[0]
st = st.sort_values(['family', 'arm'])
a.barh(range(len(st)), st.test_auc, color=[cols[f] for f in st.family],
       edgecolor='white', lw=1.2)
for i, (v, n) in enumerate(zip(st.test_auc, st.bits)):
    a.text(v + 0.0008, i, f'{v:.4f}  ({int(n):,} bits)', va='center', fontsize=8.5)
a.axvline(CEIL, ls='--', color=MUTED, lw=1.4)
a.text(CEIL, len(st) - 0.3, f' best tree {CEIL:.4f}', fontsize=9, color=MUTED)
a.set_yticks(range(len(st))); a.set_yticklabels(st.arm, fontsize=8.5)
a.set_xlim(min(st.test_auc.min() - 0.01, CEIL - 0.02), st.test_auc.max() + 0.012)
a.set_xlabel('test AUC'); a.invert_yaxis()
a.set_title('Feature regularisation on a logistic-regression head (bits only)',
            fontweight='bold', loc='left')
a.grid(alpha=.25, axis='x')
from matplotlib.patches import Patch
a.legend(handles=[Patch(color=c, label=f) for f, c in cols.items()
                  if f in set(st.family)], fontsize=8.5, frameon=False, loc='lower right')
b = ax[1]
for d in sorted(glob.glob('results/featreg/[CDEF]*')):
    fj = [f for f in glob.glob(d + '/fusion_*.json') if 'epoch' not in f and 'batch' not in f]
    if not fj: continue
    s = json.load(open(fj[0]))
    prof = s.get('flip_per_depth')
    if not prof: continue
    ks = sorted(int(k) for k in prof)
    b.plot(ks, [prof[str(k)] for k in ks], 'o-', lw=1.6, ms=5,
           label=os.path.basename(d), alpha=.85)
b.set_xlabel('tree depth of the split'); b.set_ylabel('flip probability p')
b.set_title('What each arm actually does to each layer  (p = 0 means deleted or untouched)',
            fontweight='bold', loc='left')
b.legend(fontsize=7.5, frameon=False, ncol=2); b.grid(alpha=.25)
plt.tight_layout(); plt.savefig(f'{DRIVE}/featreg_overview.png', dpi=150); plt.show()

In [ ]:
# 12 · training curves for the arms worth a close look
from plot_training import plot_training_curves
for arm in ['B_wd_' + WD, 'C_unif_0.2', 'D_ramp_0.2', 'E4_dropL45_ramp02', 'F1_keepL012']:
    p_ = f'results/featreg/{arm}'
    if os.path.isdir(p_):
        plot_training_curves(p_, model='tree', out=f'{DRIVE}/{arm}_curves.png')